# 07 · Lightning — M×N GPU

`TorchDistributor(num_processes=M*N, local_mode=False).run(fit, ...)`로 worker 노드에 분산 spawn하고, 각 child에서 `Trainer(devices=N, num_nodes=M, strategy='ddp')`로 학습합니다.

> 1×1은 [`05-launch_lightning_trainer_1x1.ipynb`](05-launch_lightning_trainer_1x1.ipynb), 1×N은 [`06-launch_lightning_trainer_1xN.ipynb`](06-launch_lightning_trainer_1xN.ipynb)에서 다룹니다.

**클러스터**: Multi-node Classic GPU, DBR 17.3 LTS ML, Workers `g5.12xlarge` × M. **Autoscaling OFF 필수**.

In [ ]:
%run ./00-setup

## 학습 함수 import

In [ ]:
import os
import sys

NOTEBOOK_PATH = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
SCRIPT_DIR = "/Workspace" + os.path.dirname(NOTEBOOK_PATH)
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)
print(f"SCRIPT_DIR={SCRIPT_DIR}")


def td_lit_fit(**kwargs):
    """TorchDistributor child에 보낼 Lightning fit() thin wrapper. inline 함수로
    by-value pickling을 받아 child에서 sys.path를 보강한 뒤 lazy import."""
    import sys
    sd = kwargs.get("script_dir")
    if sd and sd not in sys.path:
        sys.path.insert(0, sd)
    from lightning_trainer import fit
    return fit(**kwargs)

## M×N GPU

In [ ]:
import mlflow
from pyspark.ml.torch.distributor import TorchDistributor

NUM_NODES = 2          # M
NUM_GPUS_PER_NODE = 4  # N

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="lit-MxN", log_system_metrics=True) as run:
    run_id = run.info.run_id

TorchDistributor(
    num_processes=NUM_NODES * NUM_GPUS_PER_NODE,
    local_mode=False,
    use_gpu=True,
).run(
    td_lit_fit,
    experiment_path=EXPERIMENT_PATH,
    run_id=run_id,
    db_host=DB_HOST,
    db_token=DB_TOKEN,
    data_dir=DATA_DIR,
    ckpt_dir=os.path.join(CKPT_DIR, "lit-MxN"),
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    emb_dim=cfg["emb_dim"],
    tower_hidden=cfg["tower_hidden"],
    batch_size=cfg["batch_size_per_gpu"],
    num_epochs=cfg["num_epochs"],
    max_steps_per_epoch=cfg["max_steps_per_epoch"],
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    devices=NUM_GPUS_PER_NODE,
    num_nodes=NUM_NODES,
    topology="MxN",
    script_dir=SCRIPT_DIR,
)